# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-39: Respiratory rate from video - development on VitalVideos-WW's training split

### What this notebook does
Develops and screens ways of reading respiratory rate from video on VitalVideos-WW's training split, before a single pre-registered test (NB_P3_40) reads the held-out split. It first checks the breathing reference without a camera (C0), then three families of camera readouts: the motion of the torso in the stored whole-frame stream (C1), the motion of the head in the face crop (C2), and the breathing modulation of the colour and pulse signals (C3), with Rice iPPG's paced deep breathing as a positive control for the last (C4). This is the development tier: each section's method and its go/no-go are fixed before it runs, but its results may steer the design, and a readout that fails here is reported as an exploratory negative. No frame of the held-out split is read, no frame is rendered, nothing trains, and results appear as aggregates only.

### Sets
- **VitalVideos-WW, training split (VV-240)** - 240 of the 300 participants, one 60 s recording each, seated and still; lossless lab video at 60 fps whose framing includes face, chest and shoulders (NB_P3_27). Frames come from the store: `frames` (72x72 face crops at 30 fps), `frames_60`, and `frames_full` (the whole frame at 132x120 and 10 fps).
- **VitalVideos-WW, held-out split (VV-60)** - 60 participants. Its reference values are read here, camera-free; its frames are read only by NB_P3_40.
- **Rice iPPG, deep breathing** - 14 recordings paced at 6 s in and 5 s out (about 5.45 breaths/min), with face-only frames and a finger PPG; descriptive, in C4 only.

### The reference
The respiration belt worn at the waist, recorded at 500 Hz on the same clock as the ECG and the contact PPG, read as `rr_belt`: the spectral peak of the whole minute within 6-42 breaths/min, band-edge peaks refused (NB_P3_27). A belt rate is scored only where an independent surrogate agrees with it within 1 breath/min - ECG-derived respiration (the breathing modulation of R-wave amplitude) or the contact PPG's pulse amplitude - which `resp_ok` records: 264 of 300 recordings, 208 of them in VV-240 and 56 in VV-60. The recordings without a confirmed belt are reported apart and never scored.

### The measure, the development gate and the bar
- **Measure.** One rate per recording over the full 60 s, and the share within 2 breaths/min of `rr_belt` among the recordings a readout reads, always beside coverage (the share it reads). Within 1 breath/min, the mean absolute error, bias and limits of agreement are reported beside it; strata (rate band, site group, Fitzpatrick type, age) are descriptive.
- **The naive baseline.** The median belt rate of VV-240's `resp_ok` recordings, given to every recording.
- **The development gate**, fixed on 2026-11-04 before any section ran. A readout goes forward to NB_P3_40 only if, over VV-240's 208 `resp_ok` recordings, at least 82% of those it reads are within 2 breaths/min, it reads at least 80% of them, and it beats the baseline in a paired comparison (exact McNemar test, p < 0.01, with recordings it does not read counted as misses). A readout that beats the baseline below 82% is a *partial signal*; anything else is a development negative.
- **The bar NB_P3_40 applies**, fixed the same day. On VV-60's 56 `resp_ok` recordings: the Wilson 95% lower bound of the within-2 share at least 70% (46 of 56 when all are read), coverage at least 80%, and better than the baseline by exact McNemar at p < 0.05. The gate's 82% is the observed share that bar needs.

### On record before this notebook
- **NB_P3_01** read respiratory rate from MCD-rPPG's 1D store against a clinical label and a stand ECG, neither recorded with the video; its result was revised to inconclusive on 2026-11-04.
- **Respiratory sinus arrhythmia from beat intervals** (2026-10-30, exploratory): on 162 VV-240 `resp_ok` recordings, the rate of the beat intervals' breathing swing lay within 2 breaths/min of the belt in 60.5% for the ECG, 63.0% for the contact PPG, 34.8% for PhysNet v2 and 23.5% for POS, against 35.8% for a constant at the training median. C3 re-runs it.
- **A probe** (2026-11-03, exploratory) read nine VV-240 recordings - the first three `resp_ok` recordings by GUID in each site group, none with a staff-flagged belt. Within 1 breath/min of the belt: the torso below the face box in `frames_full` in 8 of 9, the head's motion in the face crop in 3, the crop's green channel in 0. C1-C3's methods were chosen after it, so those nine are reported beside the rest.
- **VV-60's belt values** were read on 2026-11-03 to score the baseline there; no VV-60 frame has been read.

### Background
Breathing reaches a camera in three ways: the chest and shoulders rise and fall, the head moves with them, and the skin's colour signal is modulated - its baseline, its pulse amplitude and its pulse rate. A 60 s recording holds one rate, between 6 and 30 or so breaths, with spectral bins 1 breath/min apart over the minute and 2 over half of it. Resting rates cluster, so a constant scores well within 2 breaths/min, and every readout here is judged against that constant rather than against chance.

In [1]:
import hashlib
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import scipy
from scipy.interpolate import interp1d
from scipy.signal import butter, detrend, filtfilt, find_peaks, resample_poly

# Walk up from the notebook until the repo root is found, identified by the MediaPipe
# asset, as in NB_P3_27 (cwd is the nested notebook folder).
MP_REL = Path('app/live_hr_demo/models/mediapipe/face_landmarker.task')
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / MP_REL).exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'no {MP_REL} in any parent of {Path.cwd()}')

VV_STORE = Path(r'D:\QualityPhys\phase 3 datasets\vitalvideos_phase3_rc_none.h5')
SPLIT_CSV = REPO_ROOT / 'Data' / 'vitalvideos_split.csv'  # GUID and split, frozen by NB_P3_27

# NB_P3_27's reference constants, copied so that its readouts reproduce the stored rates.
BIO_FS = 500.0  # belt, ECG and contact PPG
SEQ_MS = 2.0  # BIO timestamp = SEQ x 2 ms
PPG_WORK_FS = 50.0
CARDIAC_LOW_HZ, CARDIAC_HIGH_HZ = 0.7, 3.5
HR_MIN_BPM, HR_MAX_BPM = 35.0, 200.0
RESP_LOW_HZ, RESP_HIGH_HZ = 0.10, 0.70  # 6-42 breaths/min
RESP_WORK_FS = 10.0  # the belt is decimated 50x before its spectrum
SERIES_FS = 4.0  # beat-to-beat series are resampled onto this grid

# The measure and the development gate, fixed 2026-11-04.
WITHIN_BPM = 2.0
GATE_SHARE, GATE_COVERAGE, GATE_P = 0.82, 0.80, 0.01
HALF_S = 30.0  # C0 reads each half of the minute alone
PARITY_BPM = 0.01  # recomputed reference rates must equal the stored ones to this

frozen = pd.read_csv(SPLIT_CSV).set_index('guid').split
with h5py.File(VV_STORE, 'r') as store:
    in_store = pd.Series({guid: str(store[guid].attrs['split']) for guid in store})
if not in_store.sort_index().equals(frozen.sort_index()):
    raise RuntimeError(f'the store\'s split attributes differ from {SPLIT_CSV.name}')
TRAIN = sorted(in_store.index[in_store == 'train'])


def read_frames(group, name):
    """The only route by which this notebook reads frames: refuses any recording outside VV-240."""
    if str(group.attrs['split']) != 'train':
        raise RuntimeError(f'{group.name}: held-out frames are read only by NB_P3_40')
    return group[name][:]


print(f'repo {REPO_ROOT} | store {VV_STORE.stat().st_size / 1e9:.1f} GB, {len(in_store)} recordings, '
      f'split as frozen: {dict(in_store.value_counts())}')
print(f'numpy {np.__version__}, scipy {scipy.__version__}, pandas {pd.__version__}, h5py {h5py.__version__}')

repo D:\code\QualityPhys - CRVSE Project | store 34.2 GB, 300 recordings, split as frozen: {'train': np.int64(240), 'val': np.int64(60)}
numpy 2.5.0, scipy 1.18.0, pandas 3.0.3, h5py 3.16.0


## 1. C0 - The reference without a camera
The belt, ECG-derived and pulse-amplitude rates recomputed from the store's own 500 Hz signals, over the whole minute and over each half of it.

The section decides nothing about a camera readout. It answers three things the later sections rest on: whether the store reproduces the reference rates NB_P3_27 stored (a mismatch stops the notebook); how well the belt agrees with ECG-derived respiration over half a minute, and how far the belt's rate moves from one half of the minute to the other, which decides whether two half-minute readings can serve as a stability check on a camera readout; and the naive baseline in each split.

- **Readouts.** NB_P3_27's, unchanged: the belt decimated to 10 Hz and band-passed 0.05-1.0 Hz; ECG-derived respiration from R-wave amplitudes (QRS band-pass, squared and integrated over 150 ms, each peak refined to the largest deflection within 80 ms); pulse amplitude from the contact PPG at 50 Hz, systolic peak minus the preceding trough; beat-sampled series resampled to 4 Hz; every rate the zero-padded spectral peak within 6-42 breaths/min, log-parabolic refinement, band-edge peaks refused.
- **Halves.** The first and the second 30 s on the device clock (sample number x 2 ms), each read alone.
- **Parity.** Over all 300 recordings, each recomputed whole-minute rate must equal its stored attribute (`rr_belt`, `rr_edr`, `rr_ppg_amp`) within 0.01 breaths/min, with the same refusals; otherwise the section stops.
- **Reported.** By split, over all recordings and over `resp_ok`: agreement of belt and ECG-derived respiration over the minute and per half, within 1 and 2 breaths/min; the belt's change from the first half to the second; the recordings without a confirmed belt; the baseline's score; the rate bands of the early-warning score (8 or fewer, 9-11, 12-20, 21-24, 25 or more breaths/min); and a digest of the per-recording table, so that a rerun can be compared exactly.

In [2]:
# NB_P3_27's readouts (section 5), copied without change.

def bandpass(x, fs, low, high, order=2):
    '''Zero-phase Butterworth band-pass.'''
    b, a = butter(order, [low / (fs / 2), high / (fs / 2)], btype='band')
    return filtfilt(b, a, np.asarray(x, dtype=np.float64))


def resp_rate(x, fs, low=RESP_LOW_HZ, high=RESP_HIGH_HZ, pad=8):
    '''Returns the dominant respiratory rate of a series in breaths/min, or NaN.

    A zero-padded FFT with log-parabolic peak refinement, as in the app's heart-rate
    readout. A peak on either band edge is refused: it is slow drift or broadband noise
    rather than breathing, and the low edge is where drift parked several readings
    before the refusal was added.
    '''
    x = np.asarray(x, dtype=np.float64)
    if x.size < 16 or not np.all(np.isfinite(x)) or x.std() < 1e-12:
        return np.nan
    x = (x - x.mean()) * np.hanning(x.size)
    n = x.size * pad
    power = np.abs(np.fft.rfft(x, n)) ** 2
    freqs = np.fft.rfftfreq(n, 1.0 / fs)
    band = np.where((freqs >= low) & (freqs <= high))[0]
    k = int(band[np.argmax(power[band])])
    if k in (band[0], band[-1]):
        return np.nan
    left, centre, right = np.log(power[k - 1:k + 2] + 1e-30)
    denom = left - 2.0 * centre + right
    offset = 0.5 * (left - right) / denom if abs(denom) > 1e-12 else 0.0
    offset = offset if abs(offset) <= 0.5 else 0.0
    return float((k + offset) * (freqs[1] - freqs[0]) * 60.0)


def beat_series_rate(times, values):
    '''Respiratory rate of a beat-sampled series, resampled to SERIES_FS.'''
    ok = np.isfinite(values)
    times, values = np.asarray(times)[ok], np.asarray(values)[ok]
    if len(times) < 10:
        return np.nan
    grid = np.arange(times[0], times[-1], 1.0 / SERIES_FS)
    series = interp1d(times, values, kind='cubic')(grid)
    return resp_rate(detrend(series), SERIES_FS)


def belt_rate(belt):
    '''Respiratory rate of the 500 Hz belt from its spectrum.'''
    x = resample_poly(detrend(np.asarray(belt, dtype=np.float64)), 1, int(BIO_FS / RESP_WORK_FS))
    return resp_rate(bandpass(x, RESP_WORK_FS, 0.05, 1.0), RESP_WORK_FS)


def edr_rate(ecg):
    '''ECG-derived respiration: rate of the breathing modulation of R-wave amplitude.'''
    x = bandpass(detrend(np.asarray(ecg, dtype=np.float64)), BIO_FS, 5.0, 15.0, order=3)
    k = int(0.15 * BIO_FS)
    integ = np.convolve(x ** 2, np.ones(k) / k, mode='same')
    peaks, _ = find_peaks(integ, distance=int(60.0 / HR_MAX_BPM * BIO_FS),
                          height=integ.mean() + 0.5 * integ.std())
    w = int(0.08 * BIO_FS)
    r = np.array([max(0, p - w) + int(np.argmax(np.abs(x[max(0, p - w):p + w]))) for p in peaks], dtype=int)
    return beat_series_rate(r / BIO_FS, np.abs(x[r])) if len(r) else np.nan


def ppg_amplitude_rate(ppg):
    '''PPG-derived respiration: rate of the breathing modulation of pulse amplitude.'''
    x = resample_poly(detrend(np.asarray(ppg, dtype=np.float64)), 1, int(BIO_FS / PPG_WORK_FS))
    x = bandpass(x, PPG_WORK_FS, CARDIAC_LOW_HZ, CARDIAC_HIGH_HZ, order=3)
    z = (x - np.median(x)) / (1.4826 * np.median(np.abs(x - np.median(x))) + 1e-9)
    best = None
    for sign in (1.0, -1.0):
        peaks, _ = find_peaks(sign * z, distance=int(60.0 / HR_MAX_BPM * PPG_WORK_FS), prominence=0.5)
        ibi = np.diff(peaks) / PPG_WORK_FS
        if len(ibi) >= 10:
            disp = np.median(np.abs(ibi - np.median(ibi))) / np.median(ibi)
            if best is None or disp < best[0]:
                best = (disp, sign, peaks)
    if best is None:
        return np.nan
    _, sign, peaks = best
    troughs, _ = find_peaks(-sign * z, distance=int(60.0 / HR_MAX_BPM * PPG_WORK_FS), prominence=0.5)
    amp = np.full(len(peaks), np.nan)
    for i, p in enumerate(peaks):
        before = troughs[troughs < p]
        if len(before):
            amp[i] = sign * (x[p] - x[before[-1]])
    return beat_series_rate(peaks / PPG_WORK_FS, amp)

In [3]:
READERS = dict(belt=('bio_resp', belt_rate), edr=('bio_ecg', edr_rate), amp=('bio_ppg', ppg_amplitude_rate))
STORED = dict(belt='rr_belt', edr='rr_edr', amp='rr_ppg_amp')

rows = []
with h5py.File(VV_STORE, 'r') as store:
    for guid in sorted(store):
        g = store[guid]
        t = (g['bio_seq'][:].astype(np.int64) - int(g['bio_seq'][0])) * SEQ_MS / 1000.0
        parts = dict(h1=t < HALF_S, h2=(t >= HALF_S) & (t < 2 * HALF_S))
        row = dict(guid=guid, split=str(g.attrs['split']), resp_ok=bool(g.attrs['resp_ok']),
                   rr_surrogate=float(g.attrs['rr_surrogate']))
        for key, (name, reader) in READERS.items():
            x = g[name][:].astype(np.float64)
            row[f'{key}_stored'] = float(g.attrs[STORED[key]])
            row[f'{key}_full'] = reader(x)
            for part in ('h1', 'h2'):
                row[f'{key}_{part}'] = reader(x[parts[part]])
        rows.append(row)
c0 = pd.DataFrame(rows)

for key in READERS:
    recomputed, stored = c0[f'{key}_full'], c0[f'{key}_stored']
    same_refusals = bool((recomputed.isna() == stored.isna()).all())
    worst = float((recomputed - stored).abs().max())
    print(f'parity {STORED[key]:10s}: refusals identical {same_refusals}, largest difference {worst:.2e} breaths/min')
    if not same_refusals or worst > PARITY_BPM:
        raise RuntimeError(f'{STORED[key]} does not reproduce from the store')
digest = hashlib.sha256(c0.round(6).to_csv(index=False).encode('utf-8')).hexdigest()
print(f'{len(c0)} recordings | table digest {digest[:16]}')

parity rr_belt   : refusals identical True, largest difference 3.55e-15 breaths/min
parity rr_edr    : refusals identical True, largest difference 3.55e-15 breaths/min
parity rr_ppg_amp: refusals identical True, largest difference 3.55e-15 breaths/min
300 recordings | table digest 97fc5ab92d9c4eec


In [4]:
def within(a, b, tol):
    """Share of recordings where both values exist and lie within tol; and that count."""
    ok = a.notna() & b.notna()
    return float(((a - b).abs()[ok] <= tol).mean()) if ok.any() else np.nan, int(ok.sum())


scopes = {'VV-240 all': c0.split == 'train', 'VV-240 resp_ok': (c0.split == 'train') & c0.resp_ok,
          'VV-60 all': c0.split == 'val', 'VV-60 resp_ok': (c0.split == 'val') & c0.resp_ok}

print('Recordings read (not refused), whole minute / first half / second half')
for key in READERS:
    print(f'  {key:5s}' + ''.join(f' | {name} {int(c0.loc[m, f"{key}_full"].notna().sum())}/'
                                  f'{int(c0.loc[m, f"{key}_h1"].notna().sum())}/{int(c0.loc[m, f"{key}_h2"].notna().sum())} of {int(m.sum())}'
                                  for name, m in scopes.items()))

print('\nBelt against ECG-derived respiration, within 1 / within 2 breaths/min (n with both)')
for name, m in scopes.items():
    cells = []
    for part in ('full', 'h1', 'h2'):
        s1, n = within(c0.loc[m, f'belt_{part}'], c0.loc[m, f'edr_{part}'], 1.0)
        s2, _ = within(c0.loc[m, f'belt_{part}'], c0.loc[m, f'edr_{part}'], 2.0)
        cells.append(f'{part:4s} {s1:6.1%} / {s2:6.1%} (n {n})')
    print(f'  {name:15s} ' + ' | '.join(cells))

print('\nThe belt from one half of the minute to the other, resp_ok')
for name in ('VV-240 resp_ok', 'VV-60 resp_ok'):
    m = scopes[name]
    d = (c0.loc[m, 'belt_h1'] - c0.loc[m, 'belt_h2']).abs().dropna()
    e = pd.concat([(c0.loc[m, f'belt_{p}'] - c0.loc[m, 'belt_full']).abs() for p in ('h1', 'h2')]).dropna()
    print(f'  {name:15s} halves read in both {len(d)} of {int(m.sum())} | |h1 - h2| median {d.median():.2f}, '
          f'within 1 {np.mean(d <= 1):.1%}, within 2 {np.mean(d <= 2):.1%}, over 3 {np.mean(d > 3):.1%} | '
          f'|half - minute| median {e.median():.2f}, within 2 {np.mean(e <= 2):.1%}')

refused = ~c0.resp_ok
print(f'\nWithout a confirmed belt: VV-240 {int((refused & (c0.split == "train")).sum())}, '
      f'VV-60 {int((refused & (c0.split == "val")).sum())}; of those with rr_surrogate: '
      f'{int((refused & c0.rr_surrogate.notna()).sum())}')

BASELINE = float(c0.loc[scopes['VV-240 resp_ok'], 'belt_stored'].median())
print(f'\nNaive baseline: the VV-240 resp_ok median, {BASELINE:.2f} breaths/min')
for name in ('VV-240 resp_ok', 'VV-60 resp_ok'):
    err = (c0.loc[scopes[name], 'belt_stored'] - BASELINE).abs()
    print(f'  {name:15s} n {len(err)} | within 1 {np.mean(err <= 1):.1%}, within 2 {np.mean(err <= 2):.1%} '
          f'({int((err <= 2).sum())}), within 3 {np.mean(err <= 3):.1%} | mean abs error {err.mean():.2f}')

print('\nBelt rate bands, resp_ok (rates rounded to the nearest whole breath/min first)')
bands = pd.cut(c0.belt_stored.round(), [0, 8, 11, 20, 24, 99], labels=['<=8', '9-11', '12-20', '21-24', '>=25'])
table = pd.crosstab(bands[c0.resp_ok].rename('belt rate'),
                    c0.split[c0.resp_ok].map({'train': 'VV-240', 'val': 'VV-60'}).rename('split'), margins=True)
print(table.to_string())

Recordings read (not refused), whole minute / first half / second half
  belt  | VV-240 all 239/234/232 of 240 | VV-240 resp_ok 208/204/205 of 208 | VV-60 all 60/59/55 of 60 | VV-60 resp_ok 56/55/52 of 56
  edr   | VV-240 all 236/236/236 of 240 | VV-240 resp_ok 208/206/206 of 208 | VV-60 all 60/58/58 of 60 | VV-60 resp_ok 56/54/54 of 56
  amp   | VV-240 all 214/188/192 of 240 | VV-240 resp_ok 185/161/170 of 208 | VV-60 all 54/52/49 of 60 | VV-60 resp_ok 51/48/45 of 56

Belt against ECG-derived respiration, within 1 / within 2 breaths/min (n with both)
  VV-240 all      full  81.7% /  85.1% (n 235) | h1    87.0% /  89.1% (n 230) | h2    86.0% /  88.2% (n 229)
  VV-240 resp_ok  full  92.3% /  94.2% (n 208) | h1    91.1% /  92.6% (n 202) | h2    91.6% /  92.6% (n 203)
  VV-60 all       full  86.7% /  90.0% (n 60) | h1    86.2% /  87.9% (n 58) | h2    92.6% /  92.6% (n 54)
  VV-60 resp_ok   full  92.9% /  94.6% (n 56) | h1    87.0% /  87.0% (n 54) | h2    96.1% /  96.1% (n 51)

The belt fr

## 2. C0 - findings (2026-11-05)
- **The store reproduces the reference.** NB_P3_27's three whole-minute rates recompute from the store's 500 Hz signals to within 4e-15 breaths/min, with identical refusals (table digest 97fc5ab92d9c4eec), so the stored `rr_belt`, `rr_edr` and `rr_ppg_amp` are what the readouts give.
- **The belt holds over half a minute.** On VV-240's 208 `resp_ok` recordings, the belt and ECG-derived respiration agree within 2 breaths/min in 94.2% over the minute and in 92.6% in each half (within 1: 92.3%, 91.1% and 91.6%). Over all 240 the halves agree slightly better than the minute (89.1% and 88.2% against 85.1%). The pulse-amplitude surrogate reads fewer recordings, and fewer still per half (161 and 170 of the 208).
- **The breathing rate itself moves within a minute.** The belt's two halves differ by a median 1.26 breaths/min on VV-240's `resp_ok` recordings, by more than 2 in 25.6% and by more than 3 in 12.3% (VV-60: 0.87, 30.8% and 15.4%); each half lies within 2 breaths/min of the whole minute in 84.4%. ECG-derived respiration follows the belt within each half, so the change is the person's breathing, not noise in the belt.
- **Decision (2026-11-05): no half-minute stability gate.** A readout required to agree with itself across the two halves within 2 breaths/min would be refused in about a quarter of recordings even if it read every half perfectly, which alone breaks the gate's 80% coverage; within 3 it would still lose an eighth. Every readout is judged on the whole minute, and agreement between halves is described, never gated.
- **The baseline.** The median belt rate of VV-240's `resp_ok` recordings is 19.94 breaths/min. Given to every recording, it lies within 2 breaths/min in 34.1% of VV-240's 208 (71) and 39.3% of VV-60's 56 (22), within 1 in 17.3% and 21.4%, at a mean absolute error of 3.54 and 3.36 breaths/min.
- **The rates cover the range unevenly.** Rounded to whole breaths, VV-240's 208 hold 3 recordings at 8 or fewer, 7 at 9-11, 102 at 12-20, 67 at 21-24 and 29 at 25 or more; VV-60's 56 hold 3, 1, 35, 13 and 4. VV-60 holds too few at either extreme for any claim about slow or fast breathing.
- **Without a confirmed belt** are 32 VV-240 and 4 VV-60 recordings; 14 of the 36 carry `rr_surrogate`.

## 3. C1 - Torso motion in the whole frame
The breathing rhythm read from the movement of the shoulders and chest in `frames_full`, by two readouts fixed before this section ran, on all 240 VV-240 recordings.

- **Region.** In grey (the mean of R, G and B), every row from the bottom of the stored padded face box (`box_ry1`) to the bottom of the frame, all columns. A recording with fewer than 8 such rows is not read.
- **Readout A, pixel components.** Each pixel's series is detrended and band-passed to 0.08-1.0 Hz (second-order Butterworth, zero-phase); the first five principal components are each read with `resp_rate` (6-42 breaths/min, band-edge peaks refused); the readout is the rate of the component whose spectrum is most concentrated - the share of in-band power within 0.03 Hz of its peak - among those not refused.
- **Readout B, vertical shift.** Each frame's region is matched against the recording's mean region by Hann-weighted cross-correlation with a parabolic sub-pixel peak; the vertical series is detrended, band-passed the same way and read with `resp_rate`.
- **No confidence threshold.** Every rate a readout produces counts as read. Accuracy by quintile of peak concentration is reported, so that a refusal rule can be proposed at the gate if one is wanted.
- **The gate, per readout,** as fixed in the header: over the 208 `resp_ok` recordings, at least 82% of those read within 2 breaths/min of `rr_belt`, at least 80% of the 208 read, and better than the baseline by an exact McNemar test at p < 0.01, the readout ahead on the discordant pairs and an unread recording counted as a miss. A readout that beats the baseline below 82% is a partial signal; anything else is a development negative.
- **Described beside it, deciding nothing.** Within 1 breath/min, the mean absolute error, bias and 95% limits of agreement; by site group, Fitzpatrick pair (1-2, 3-4, 5-6), age band (40 or under, 41-60, over 60) and belt-rate band; the nine probe recordings of 2026-11-03 apart from the other 199; readout A against readout B; each readout on each half of the minute against the belt's same half from C0; the 32 recordings without a confirmed belt against `rr_surrogate` where it exists.
- **Seen before this section ran.** The nine-recording probe read both readouts in this form: A within 1 breath/min of the belt in 8 of 9, B in 7 of 9.

In [5]:
from scipy.signal import sosfiltfilt
from scipy.stats import binomtest

FULL_FS = 10.0  # frames_full: the whole frame at 10 fps
MIN_TORSO_ROWS = 8
N_COMPONENTS = 5
SHARE_HALF_WIDTH_HZ = 0.03
BREATH_SOS = butter(2, [0.08 / (FULL_FS / 2), 1.0 / (FULL_FS / 2)], btype='band', output='sos')


def torso_region(frames_full, box_ry1):
    """Grey rows from the bottom of the padded face box to the bottom of the frame, or None."""
    grey = frames_full.astype(np.float64).mean(axis=3)
    top = int(np.ceil(box_ry1 * grey.shape[1]))
    return grey[:, top:, :] if grey.shape[1] - top >= MIN_TORSO_ROWS else None


def breathing_band(x):
    """Linear detrend and the 0.08-1.0 Hz band-pass, along time."""
    return sosfiltfilt(BREATH_SOS, detrend(x, axis=0), axis=0)


def peak_share(x, fs, pad=8):
    """Share of the 6-42 breaths/min power within SHARE_HALF_WIDTH_HZ of its peak, on resp_rate's spectrum."""
    x = np.asarray(x, dtype=np.float64)
    x = (x - x.mean()) * np.hanning(x.size)
    power = np.abs(np.fft.rfft(x, x.size * pad)) ** 2
    freqs = np.fft.rfftfreq(x.size * pad, 1.0 / fs)
    band = (freqs >= RESP_LOW_HZ) & (freqs <= RESP_HIGH_HZ)
    peak = freqs[band][np.argmax(power[band])]
    return float(power[band & (np.abs(freqs - peak) <= SHARE_HALF_WIDTH_HZ)].sum() / power[band].sum())


def components_rate(region):
    """Readout A: (rate, share, component) of the most concentrated of the first principal
    components of the pixels' breathing-band series; NaNs when every component is refused.

    The components' time series come from the frames' Gram matrix: the same U x S as a
    singular value decomposition of the pixel matrix, up to sign, at about a sixth of its cost
    when the pixels far outnumber the frames.
    """
    X = breathing_band(region.reshape(len(region), -1))
    X = X - X.mean(axis=0)
    values, vectors = np.linalg.eigh(X @ X.T)
    best = (np.nan, np.nan, -1)
    for rank, j in enumerate(np.argsort(values)[::-1][:N_COMPONENTS]):
        series = vectors[:, j] * np.sqrt(max(values[j], 0.0))
        rate = resp_rate(series, FULL_FS)
        if np.isfinite(rate):
            share = peak_share(series, FULL_FS)
            if not np.isfinite(best[1]) or share > best[1]:
                best = (rate, share, rank)
    return best


def vertical_shift(region):
    """Each frame's vertical offset from the recording's mean region, in pixels: Hann-weighted
    cross-correlation, its peak refined by a parabola through the neighbouring rows."""
    n, h, w = region.shape
    weight = np.outer(np.hanning(h), np.hanning(w))
    mean_region = region.mean(axis=0)
    reference = np.conj(np.fft.rfft2((mean_region - mean_region.mean()) * weight))
    dy = np.zeros(n)
    for i in range(n):
        frame = region[i]
        corr = np.fft.fftshift(np.fft.irfft2(np.fft.rfft2((frame - frame.mean()) * weight) * reference, s=(h, w)))
        iy, ix = np.unravel_index(int(corr.argmax()), corr.shape)
        iy, ix = min(max(iy, 1), h - 2), min(max(ix, 1), w - 2)
        left, centre, right = corr[iy - 1, ix], corr[iy, ix], corr[iy + 1, ix]
        curvature = left - 2.0 * centre + right
        offset = 0.5 * (left - right) / curvature if abs(curvature) > 1e-12 else 0.0
        dy[i] = iy - h // 2 + (offset if abs(offset) <= 0.5 else 0.0)
    return dy


def shift_rate(dy):
    """Readout B: (rate, share) of the vertical shift's breathing-band series."""
    series = breathing_band(dy)
    return resp_rate(series, FULL_FS), peak_share(series, FULL_FS)

In [6]:
rows = []
with h5py.File(VV_STORE, 'r') as store:
    for guid in TRAIN:
        g = store[guid]
        a = g.attrs
        region = torso_region(read_frames(g, 'frames_full'), float(a['box_ry1']))
        row = dict(guid=guid, resp_ok=bool(a['resp_ok']), note_resp_bad=bool(a['note_resp_bad']),
                   site_group=str(a['site_group']), fitzpatrick=int(a['fitzpatrick']), age=int(a['age']),
                   belt=float(a['rr_belt']), rr_surrogate=float(a['rr_surrogate']),
                   torso_rows=0 if region is None else region.shape[1])
        if region is not None:
            row['a_rate'], row['a_share'], row['a_component'] = components_rate(region)
            dy = vertical_shift(region)
            row['b_rate'], row['b_share'] = shift_rate(dy)
            half = len(region) // 2
            for part, part_slice in (('h1', slice(0, half)), ('h2', slice(half, 2 * half))):
                row[f'a_{part}'] = components_rate(region[part_slice])[0]
                row[f'b_{part}'] = shift_rate(dy[part_slice])[0]
        rows.append(row)
c1 = pd.DataFrame(rows).merge(c0[['guid', 'belt_h1', 'belt_h2']], on='guid', how='left')

# The 2026-11-03 probe's nine: the first three resp_ok recordings by GUID per site group,
# none with a staff-flagged belt.
eligible = c1[c1.resp_ok & ~c1.note_resp_bad].sort_values('guid')
c1['probe'] = c1.guid.isin(eligible.groupby('site_group').head(3).guid)
digest = hashlib.sha256(c1.round(6).to_csv(index=False).encode('utf-8')).hexdigest()
print(f'{len(c1)} recordings, {int(c1.probe.sum())} from the probe | torso rows {int(c1.torso_rows.min())}-'
      f'{int(c1.torso_rows.max())} (median {c1.torso_rows.median():.0f}) | table digest {digest[:16]}')

240 recordings, 9 from the probe | torso rows 11-61 (median 39) | table digest aaa38673f8577b7f


In [7]:
def wilson(k, n, z=1.959964):
    """Wilson score interval for k successes in n trials."""
    if n == 0:
        return np.nan, np.nan
    p, d = k / n, 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return centre - half, centre + half


def hits(frame, col, reference='belt', tol=WITHIN_BPM):
    """(read, within tol) masks for one readout against a reference column."""
    read = frame[col].notna() & frame[reference].notna()
    return read, read & ((frame[col] - frame[reference]).abs() <= tol)


scored = c1[c1.resp_ok].copy()
base_ok = (scored.belt - BASELINE).abs() <= WITHIN_BPM
READOUTS = {'A pixel components': 'a_rate', 'B vertical shift': 'b_rate'}

print(f'The gate, over VV-240\'s {len(scored)} resp_ok recordings (baseline {BASELINE:.2f} breaths/min, '
      f'within 2 in {int(base_ok.sum())})')
for name, col in READOUTS.items():
    read, ok = hits(scored, col)
    n_read, k = int(read.sum()), int(ok.sum())
    share, coverage = (k / n_read if n_read else np.nan), n_read / len(scored)
    lo, hi = wilson(k, n_read)
    b, c = int((ok & ~base_ok).sum()), int((~ok & base_ok).sum())
    p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
    beats = b > c and p < GATE_P
    verdict = ('goes forward' if beats and share >= GATE_SHARE and coverage >= GATE_COVERAGE
               else 'partial signal' if beats and share < GATE_SHARE else 'development negative')
    err = (scored.loc[read, col] - scored.loc[read, 'belt'])
    within1 = int((err.abs() <= 1.0).sum())
    print(f'  {name}: read {n_read} of {len(scored)} ({coverage:.1%}) | within 2 {k} ({share:.1%}, 95% {lo:.1%}-{hi:.1%}) | '
          f'within 1 {within1} ({within1 / n_read:.1%}) | mean abs error {err.abs().mean():.2f}, bias {err.mean():+.2f}, '
          f'limits of agreement {err.mean() - 1.96 * err.std():+.2f} to {err.mean() + 1.96 * err.std():+.2f}')
    print(f'      McNemar against the baseline: readout only {b}, baseline only {c}, p = {p:.2g} -> {verdict}')

fitz_pair = scored.fitzpatrick.map({1: '1-2', 2: '1-2', 3: '3-4', 4: '3-4', 5: '5-6', 6: '5-6'})
age_band = pd.cut(scored.age, [0, 40, 60, 120], labels=['<=40', '41-60', '>60'])
rate_band = pd.cut(scored.belt.round(), [0, 8, 11, 20, 24, 99], labels=['<=8', '9-11', '12-20', '21-24', '>=25'])
probe = scored.probe.map({True: 'probe nine', False: 'other 199'})
STRATA = (('site group', scored.site_group, sorted(scored.site_group.unique())),
          ('Fitzpatrick', fitz_pair, ['1-2', '3-4', '5-6']),
          ('age', age_band, ['<=40', '41-60', '>60']),
          ('belt rate', rate_band, ['<=8', '9-11', '12-20', '21-24', '>=25']),
          ('probe', probe, ['probe nine', 'other 199']))
print('\nWithin 2 breaths/min among those read (right/read, of n), described')
for label, groups, keys in STRATA:
    cells = []
    for key in keys:
        sub = scored[groups == key]
        parts = []
        for col in READOUTS.values():
            read, ok = hits(sub, col)
            parts.append(f'{int(ok.sum())}/{int(read.sum())}')
        cells.append(f'{key} n {len(sub)}: A {parts[0]}, B {parts[1]}')
    print(f'  {label:11s} ' + ' | '.join(cells))

print('\nWithin 2 by quintile of peak concentration (lowest first), among those read')
for name, col, share_col in (('A', 'a_rate', 'a_share'), ('B', 'b_rate', 'b_share')):
    read, ok = hits(scored, col)
    quintile = pd.qcut(scored.loc[read, share_col], 5, labels=False)
    summary = [f'{int(ok[read][quintile == q].sum())}/{int((quintile == q).sum())} '
               f'(share {scored.loc[read, share_col][quintile == q].min():.2f}+)' for q in range(5)]
    print(f'  {name}: ' + ' | '.join(summary))

both = scored.a_rate.notna() & scored.b_rate.notna()
a_ok, b_ok = hits(scored, 'a_rate')[1], hits(scored, 'b_rate')[1]
print(f'\nA against B, both read in {int(both.sum())}: within 2 of each other '
      f'{int(((scored.a_rate - scored.b_rate).abs()[both] <= WITHIN_BPM).sum())} | both right {int((a_ok & b_ok).sum())}, '
      f'A only {int((a_ok & ~b_ok).sum())}, B only {int((~a_ok & b_ok).sum())}, neither {int((both & ~a_ok & ~b_ok).sum())}')

print('\nEach half of the minute against the belt\'s same half (C0), within 2: read and right')
for name, prefix in (('A', 'a'), ('B', 'b')):
    parts = []
    for part in ('h1', 'h2'):
        read, ok = hits(scored, f'{prefix}_{part}', reference=f'belt_{part}')
        parts.append(f'{part} {int(ok.sum())}/{int(read.sum())}')
    print(f'  {name}: ' + ', '.join(parts))

unconfirmed = c1[~c1.resp_ok]
with_surrogate = unconfirmed[unconfirmed.rr_surrogate.notna()]
print(f'\nWithout a confirmed belt: {len(unconfirmed)}, of which {len(with_surrogate)} with rr_surrogate; within 2 of it: '
      + ', '.join(f'{name[0]} {int(hits(with_surrogate, col, reference="rr_surrogate")[1].sum())}/'
                  f'{int(hits(with_surrogate, col, reference="rr_surrogate")[0].sum())}' for name, col in READOUTS.items()))

The gate, over VV-240's 208 resp_ok recordings (baseline 19.94 breaths/min, within 2 in 71)
  A pixel components: read 208 of 208 (100.0%) | within 2 197 (94.7%, 95% 90.8%-97.0%) | within 1 185 (88.9%) | mean abs error 0.61, bias -0.32, limits of agreement -4.39 to +3.74
      McNemar against the baseline: readout only 128, baseline only 2, p = 1.3e-35 -> goes forward
  B vertical shift: read 208 of 208 (100.0%) | within 2 197 (94.7%, 95% 90.8%-97.0%) | within 1 188 (90.4%) | mean abs error 0.73, bias -0.45, limits of agreement -6.19 to +5.28
      McNemar against the baseline: readout only 128, baseline only 2, p = 1.3e-35 -> goes forward

Within 2 breaths/min among those read (right/read, of n), described
  site group  CH* n 71: A 66/71, B 67/71 | CRC/SCH/BET n 71: A 70/71, B 65/71 | MOL/RING n 66: A 61/66, B 65/66
  Fitzpatrick 1-2 n 59: A 55/59, B 58/59 | 3-4 n 68: A 67/68, B 61/68 | 5-6 n 81: A 75/81, B 78/81
  age         <=40 n 68: A 64/68, B 63/68 | 41-60 n 100: A 97/100, B 98/